In [11]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   CatBoost Regression Pipeline - Gold Price Return Prediction   ║
║   Production-Ready | No Data Leakage | Walk-Forward Validation  ║
╚══════════════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────
# 0. IMPORTS & CONFIG
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
from typing import Tuple, Dict, List, Optional

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────
# GLOBAL CONFIG
# ─────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST = remaining 0.15

# ─────────────────────────────────────────────
# 1. DATA GENERATION (synthetic gold data)
# ─────────────────────────────────────────────
def generate_gold_data(n_days: int = 2000) -> pd.DataFrame:
    """
    สร้าง synthetic gold price data ที่มีลักษณะคล้าย real-world:
    - mean-reverting trend
    - volatility clustering
    - macro factor influence
    """
    np.random.seed(SEED)
    dates = pd.date_range(start="2017-01-01", periods=n_days, freq="B")

    # Simulate gold price via GBM + mean reversion
    price = [1200.0]
    vol = 0.012
    for i in range(1, n_days):
        shock = np.random.normal(0, vol)
        mean_rev = 0.001 * (1300 - price[-1]) / 1300
        price.append(price[-1] * (1 + mean_rev + shock))

    df = pd.DataFrame({"date": dates, "close": price})
    df["close"] = df["close"].clip(lower=800)

    # Synthetic macro features
    df["usd_index"]    = 100 + np.cumsum(np.random.normal(0, 0.3, n_days))
    df["real_yield"]   = np.sin(np.linspace(0, 6*np.pi, n_days)) * 1.5
    df["vix"]          = np.abs(np.random.normal(20, 5, n_days))
    df["oil_price"]    = 60 + np.cumsum(np.random.normal(0, 0.5, n_days))
    df["volume_proxy"] = np.abs(np.random.normal(1e6, 2e5, n_days))

    return df.set_index("date")


# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING (No Leakage)
# ─────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    สร้าง features ทั้งหมดโดยใช้ .shift(1) เพื่อป้องกัน data leakage
    ทุก feature ต้องรู้ก่อน t (ใช้ข้อมูล t-1, t-2, ...)
    """
    d = df.copy()

    # ── TARGET: log return วันถัดไป ──
    d["target"] = np.log(d["close"] / d["close"].shift(1)).shift(-1)

    # ── PRICE FEATURES (lag 1 day to avoid leakage) ──
    d["log_ret_1d"]  = np.log(d["close"] / d["close"].shift(1))
    d["log_ret_2d"]  = np.log(d["close"] / d["close"].shift(2))
    d["log_ret_5d"]  = np.log(d["close"] / d["close"].shift(5))
    d["log_ret_10d"] = np.log(d["close"] / d["close"].shift(10))
    d["log_ret_21d"] = np.log(d["close"] / d["close"].shift(21))

    # ── ROLLING STATS (shift to avoid leakage) ──
    for w in [5, 10, 21, 63]:
        d[f"ma_{w}"]        = d["close"].rolling(w).mean().shift(1)
        d[f"std_{w}"]       = d["close"].rolling(w).std().shift(1)
        d[f"ret_std_{w}"]   = d["log_ret_1d"].rolling(w).std().shift(1)
        d[f"ret_skew_{w}"]  = d["log_ret_1d"].rolling(w).skew().shift(1)
        d[f"ret_kurt_{w}"]  = d["log_ret_1d"].rolling(w).kurt().shift(1)

    # ── MOMENTUM ──
    d["rsi_14"] = _rsi(d["close"], 14)
    d["rsi_28"] = _rsi(d["close"], 28)
    d["macd"]   = (d["close"].ewm(span=12).mean() - d["close"].ewm(span=26).mean()).shift(1)
    d["macd_signal"] = d["macd"].ewm(span=9).mean().shift(1)
    d["macd_hist"]   = (d["macd"] - d["macd_signal"]).shift(1)

    # ── VOLATILITY ──
    d["atr_14"] = _atr(d["close"], 14)
    d["realized_vol_21"] = d["log_ret_1d"].rolling(21).std().shift(1) * np.sqrt(252)

    # ── PRICE LEVEL ──
    d["dist_from_ma21"] = ((d["close"] / d["ma_21"]) - 1).shift(1)
    d["dist_from_ma63"] = ((d["close"] / d["ma_63"]) - 1).shift(1)
    d["price_vs_52wk_high"] = (d["close"] / d["close"].rolling(252).max()).shift(1)

    # ── MACRO FEATURES (already lag-aware in real scenario) ──
    for col in ["usd_index", "real_yield", "vix", "oil_price"]:
        d[f"{col}_lag1"]   = d[col].shift(1)
        d[f"{col}_ret5d"]  = (d[col] / d[col].shift(5) - 1).shift(1)
        d[f"{col}_ret21d"] = (d[col] / d[col].shift(21) - 1).shift(1)

    # ── VOLUME ──
    d["vol_ratio_5_21"] = (d["volume_proxy"].rolling(5).mean() /
                           d["volume_proxy"].rolling(21).mean()).shift(1)

    # ── CALENDAR FEATURES ──
    d["day_of_week"]  = d.index.dayofweek
    d["month"]        = d.index.month
    d["quarter"]      = d.index.quarter
    d["is_month_end"] = d.index.is_month_end.astype(int)

    # ── INTERACTION FEATURES ──
    d["vol_x_usd"]     = d["realized_vol_21"] * d["usd_index_lag1"]
    d["vix_x_ret5d"]   = d["vix_lag1"] * d["log_ret_5d"]

    # Drop rows with NaN (from rolling windows)
    d = d.dropna()

    return d


def _rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-10)
    return (100 - 100 / (1 + rs)).shift(1)


def _atr(close: pd.Series, period: int = 14) -> pd.Series:
    tr = close.diff().abs()
    return tr.rolling(period).mean().shift(1)


# ─────────────────────────────────────────────
# 3. DATA SPLIT (Time-Based, No Shuffle)
# ─────────────────────────────────────────────
def time_based_split(df: pd.DataFrame, target_col: str = "target"):
    """
    แบ่ง data แบบ time-based เพื่อป้องกัน look-ahead bias
    train → val → test (เรียงตามเวลา ไม่ shuffle)
    """
    n = len(df)
    train_end = int(n * TRAIN_RATIO)
    val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

    feature_cols = [c for c in df.columns if c not in [target_col, "close",
                    "usd_index", "real_yield", "vix", "oil_price", "volume_proxy"]]

    X = df[feature_cols]
    y = df[target_col]

    X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
    X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
    X_test,  y_test  = X.iloc[val_end:], y.iloc[val_end:]

    print(f"\n{'─'*55}")
    print(f"  DATA SPLIT SUMMARY")
    print(f"{'─'*55}")
    print(f"  Train: {X_train.index[0].date()} → {X_train.index[-1].date()}  ({len(X_train):>4} rows)")
    print(f"  Val:   {X_val.index[0].date()}   → {X_val.index[-1].date()}    ({len(X_val):>4} rows)")
    print(f"  Test:  {X_test.index[0].date()}  → {X_test.index[-1].date()}   ({len(X_test):>4} rows)")
    print(f"  Features: {len(feature_cols)}")
    print(f"  Target std (train): {y_train.std():.6f}")
    print(f"{'─'*55}\n")

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols


# ─────────────────────────────────────────────
# 4. METRICS
# ─────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label: str = "") -> Dict:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    ic   = corr  # Information Coefficient

    # Hit Rate (directional accuracy)
    hit_rate = np.mean(np.sign(y_true) == np.sign(y_pred))

    metrics = {"RMSE": rmse, "MAE": mae, "IC": ic, "HitRate": hit_rate}

    if label:
        print(f"  [{label}] RMSE={rmse:.6f} | MAE={mae:.6f} | IC={ic:.4f} | HitRate={hit_rate:.3f}")

    return metrics


# ─────────────────────────────────────────────
# 5. BASELINE MODEL
# ─────────────────────────────────────────────
def train_baseline(X_train, y_train, X_val, y_val) -> CatBoostRegressor:
    """Train baseline CatBoost with conservative defaults"""

    params = {
        "iterations":    1000,
        "depth":         6,
        "learning_rate": 0.05,
        "l2_leaf_reg":   3.0,
        "random_seed":   SEED,
        "eval_metric":   "RMSE",
        "loss_function": "RMSE",
        "verbose":       False,
        "early_stopping_rounds": 50,
    }

    model = CatBoostRegressor(**params)
    train_pool = Pool(X_train, y_train)
    val_pool   = Pool(X_val, y_val)

    model.fit(train_pool, eval_set=val_pool, verbose=False)

    print(" Baseline Model:")
    compute_metrics(y_train, model.predict(X_train), "Train")
    compute_metrics(y_val,   model.predict(X_val),   "Val  ")

    return model


# ─────────────────────────────────────────────
# 6. HYPERPARAMETER TUNING (Optuna)
# ─────────────────────────────────────────────
def optuna_objective(trial, X_train, y_train, n_splits: int = 5):
    """
    Objective function สำหรับ Optuna
    ใช้ TimeSeriesSplit เพื่อป้องกัน leakage ระหว่าง CV
    """
    # ── Search Space ──
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bernoulli", "Bayesian", "MVS"])

    params = {
        "iterations":    trial.suggest_int("iterations", 300, 1500),
        "depth":         trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "l2_leaf_reg":   trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),
        "bootstrap_type": bootstrap_type,
        "random_seed": SEED,
        "eval_metric": "RMSE",
        "loss_function": "RMSE",
        "verbose": False,
        "early_stopping_rounds": 30,
    }

    # Subsample: เฉพาะ Bernoulli และ MVS เท่านั้น (Bayesian ไม่รองรับ)
    if bootstrap_type in ["Bernoulli", "MVS"]:
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    # Bayesian ใช้ bagging_temperature แทน
    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 1.0)

    # TimeSeriesSplit CV (ไม่ shuffle)
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_vl = X_train.iloc[val_idx]
        y_vl = y_train.iloc[val_idx]

        m = CatBoostRegressor(**params)
        m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_vl, y_vl), verbose=False)

        preds  = m.predict(X_vl)
        rmse   = np.sqrt(mean_squared_error(y_vl, preds))
        scores.append(rmse)

        # Optuna pruning
        trial.report(np.mean(scores), step=len(scores))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


def run_optuna_tuning(X_train, y_train, n_trials: int = 50) -> Dict:
    """Run Optuna study with MedianPruner"""

    pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=3)
    study  = optuna.create_study(direction="minimize", pruner=pruner,
                                  sampler=optuna.samplers.TPESampler(seed=SEED))

    study.optimize(
        lambda trial: optuna_objective(trial, X_train, y_train),
        n_trials=n_trials,
        show_progress_bar=False,
    )

    best = study.best_params
    print(f"\nBest Params (RMSE={study.best_value:.6f}):")
    for k, v in best.items():
        print(f"   {k}: {v}")

    return best, study


# ─────────────────────────────────────────────
# 7. TRAIN TUNED MODEL
# ─────────────────────────────────────────────
def train_tuned_model(best_params: Dict, X_train, y_train,
                      X_val, y_val) -> CatBoostRegressor:
    """Train final model with best hyperparameters"""

    params = {**best_params,
              "random_seed":   SEED,
              "eval_metric":   "RMSE",
              "loss_function": "RMSE",
              "verbose":       False,
              "early_stopping_rounds": 50}

    model = CatBoostRegressor(**params)
    model.fit(Pool(X_train, y_train), eval_set=Pool(X_val, y_val), verbose=False)

    return model


# ─────────────────────────────────────────────
# 8. WALK-FORWARD VALIDATION
# ─────────────────────────────────────────────
def walk_forward_validation(df_features: pd.DataFrame,
                            best_params: Dict,
                            target_col: str = "target",
                            n_folds: int = 5) -> pd.DataFrame:
    """
    Expanding window walk-forward validation
    แต่ละ fold: train บน data ที่ผ่านมาทั้งหมด, test บน window ถัดไป
    """
    feature_cols = [c for c in df_features.columns
                    if c not in [target_col, "close",
                                 "usd_index", "real_yield", "vix",
                                 "oil_price", "volume_proxy"]]

    X = df_features[feature_cols]
    y = df_features[target_col]

    n = len(df_features)
    fold_size = n // (n_folds + 1)
    min_train = fold_size * 2  # minimum 2 folds for training

    results = []

    params = {**best_params,
              "random_seed":   SEED,
              "eval_metric":   "RMSE",
              "loss_function": "RMSE",
              "verbose":       False}

    print(f"\n{'─'*55}")
    print(f"  WALK-FORWARD VALIDATION ({n_folds} folds, expanding window)")
    print(f"{'─'*55}")

    for fold in range(n_folds):
        test_start = min_train + fold * fold_size
        test_end   = test_start + fold_size

        if test_end > n:
            break

        X_tr = X.iloc[:test_start]
        y_tr = y.iloc[:test_start]
        X_te = X.iloc[test_start:test_end]
        y_te = y.iloc[test_start:test_end]

        val_start = int(len(X_tr) * 0.85)
        X_vl = X_tr.iloc[val_start:]
        y_vl = y_tr.iloc[val_start:]
        X_tr_ = X_tr.iloc[:val_start]
        y_tr_ = y_tr.iloc[:val_start]

        m = CatBoostRegressor(**params, early_stopping_rounds=30)
        m.fit(Pool(X_tr_, y_tr_), eval_set=Pool(X_vl, y_vl), verbose=False)

        preds = m.predict(X_te)
        rmse  = np.sqrt(mean_squared_error(y_te, preds))
        mae   = mean_absolute_error(y_te, preds)
        ic    = np.corrcoef(y_te, preds)[0, 1]
        hr    = np.mean(np.sign(y_te) == np.sign(preds))

        print(f"  Fold {fold+1}: {X_te.index[0].date()} → {X_te.index[-1].date()} "
              f"| RMSE={rmse:.6f} | IC={ic:.4f} | HR={hr:.3f}")

        results.append({
            "fold": fold + 1,
            "start": X_te.index[0],
            "end": X_te.index[-1],
            "rmse": rmse, "mae": mae, "ic": ic, "hit_rate": hr,
            "train_size": len(X_tr_),
            "test_size": len(X_te),
        })

    results_df = pd.DataFrame(results)
    print(f"\n  Mean RMSE: {results_df['rmse'].mean():.6f} ± {results_df['rmse'].std():.6f}")
    print(f"  Mean IC:   {results_df['ic'].mean():.4f} ± {results_df['ic'].std():.4f}")
    print(f"  Mean HR:   {results_df['hit_rate'].mean():.3f}")
    print(f"{'─'*55}\n")

    return results_df


# ─────────────────────────────────────────────
# 9. ROBUSTNESS CHECK (Multiple Seeds)
# ─────────────────────────────────────────────
def robustness_check(best_params: Dict, X_train, y_train,
                     X_val, y_val, X_test, y_test,
                     seeds: List[int] = [42, 123, 456, 789, 2024]) -> pd.DataFrame:
    """ทดสอบ stability ของโมเดลด้วย multiple random seeds"""

    results = []
    params_base = {**best_params,
                   "eval_metric":   "RMSE",
                   "loss_function": "RMSE",
                   "verbose":       False,
                   "early_stopping_rounds": 30}

    for seed in seeds:
        params = {**params_base, "random_seed": seed}
        m = CatBoostRegressor(**params)
        m.fit(Pool(X_train, y_train), eval_set=Pool(X_val, y_val), verbose=False)

        val_rmse  = np.sqrt(mean_squared_error(y_val,  m.predict(X_val)))
        test_rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test)))
        test_ic   = np.corrcoef(y_test, m.predict(X_test))[0, 1]

        results.append({"seed": seed, "val_rmse": val_rmse,
                         "test_rmse": test_rmse, "test_ic": test_ic})

    df = pd.DataFrame(results)
    print(f"\n{'─'*55}")
    print(f"  ROBUSTNESS CHECK ({len(seeds)} seeds)")
    print(f"{'─'*55}")
    print(f"  Val  RMSE: {df['val_rmse'].mean():.6f} ± {df['val_rmse'].std():.6f}")
    print(f"  Test RMSE: {df['test_rmse'].mean():.6f} ± {df['test_rmse'].std():.6f}")
    print(f"  Test IC:   {df['test_ic'].mean():.4f} ± {df['test_ic'].std():.4f}")
    print(f"{'─'*55}\n")

    return df


# ─────────────────────────────────────────────
# 10. VISUALIZATION
# ─────────────────────────────────────────────
def create_full_report(baseline_model, tuned_model, study,
                       X_train, y_train, X_val, y_val, X_test, y_test,
                       walk_fwd_results, robustness_results,
                       feature_cols) -> None:
    """สร้าง comprehensive visualization report"""

    fig = plt.figure(figsize=(22, 26))
    fig.patch.set_facecolor("#0f1117")
    gs  = gridspec.GridSpec(5, 3, figure=fig,
                             hspace=0.45, wspace=0.35,
                             top=0.95, bottom=0.04)

    colors = {"train": "#4fc3f7", "val": "#81c784", "test": "#ffb74d",
              "pred":  "#f48fb1", "base": "#90a4ae", "tuned": "#ce93d8"}

    style = {"color": "#e0e0e0", "fontsize": 9}
    ax_bg = "#1a1d27"

    def style_ax(ax, title):
        ax.set_facecolor(ax_bg)
        ax.set_title(title, color="#e0e0e0", fontsize=10, fontweight="bold", pad=8)
        ax.tick_params(colors="#9e9e9e", labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#333")
        ax.xaxis.label.set_color("#9e9e9e")
        ax.yaxis.label.set_color("#9e9e9e")
        ax.grid(True, alpha=0.15, color="#555")

    # ── TITLE ──
    fig.text(0.5, 0.975, " CatBoost Gold Return Prediction — Full Pipeline Report",
             ha="center", va="top", color="#ffffff",
             fontsize=15, fontweight="bold")

    # ── 1. Target Distribution ──
    ax1 = fig.add_subplot(gs[0, 0])
    y_all = pd.concat([y_train, y_val, y_test])
    ax1.hist(y_train, bins=50, alpha=0.7, color=colors["train"], label="Train", density=True)
    ax1.hist(y_val,   bins=50, alpha=0.7, color=colors["val"],   label="Val",   density=True)
    ax1.hist(y_test,  bins=50, alpha=0.7, color=colors["test"],  label="Test",  density=True)
    ax1.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax1, " Target Distribution (Log Returns)")
    ax1.set_xlabel("Log Return")

    # ── 2. Baseline vs Tuned: Scatter Val ──
    ax2 = fig.add_subplot(gs[0, 1])
    base_pred_val  = baseline_model.predict(X_val)
    tuned_pred_val = tuned_model.predict(X_val)
    ax2.scatter(y_val, base_pred_val,  alpha=0.3, s=8, color=colors["base"],  label="Baseline")
    ax2.scatter(y_val, tuned_pred_val, alpha=0.3, s=8, color=colors["tuned"], label="Tuned")
    mn, mx = y_val.min(), y_val.max()
    ax2.plot([mn, mx], [mn, mx], "w--", alpha=0.4, lw=1)
    ax2.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax2, " Predicted vs Actual (Val Set)")
    ax2.set_xlabel("Actual"); ax2.set_ylabel("Predicted")

    # ── 3. Optuna Trial History ──
    ax3 = fig.add_subplot(gs[0, 2])
    trial_vals = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.minimum.accumulate(trial_vals)
    ax3.plot(trial_vals, alpha=0.4, color=colors["val"], lw=1, label="Trial RMSE")
    ax3.plot(best_so_far, color=colors["tuned"], lw=2, label="Best so far")
    ax3.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax3, " Optuna Trial History")
    ax3.set_xlabel("Trial"); ax3.set_ylabel("CV RMSE")

    # ── 4. Feature Importance (top 15) ──
    ax4 = fig.add_subplot(gs[1, :2])
    fi = pd.Series(tuned_model.get_feature_importance(),
                   index=feature_cols).sort_values(ascending=True).tail(15)
    colors_fi = plt.cm.plasma(np.linspace(0.3, 0.9, len(fi)))
    bars = ax4.barh(fi.index, fi.values, color=colors_fi)
    style_ax(ax4, " Top 15 Feature Importance (Tuned Model)")
    ax4.set_xlabel("Importance Score")

    # ── 5. Walk-Forward RMSE ──
    ax5 = fig.add_subplot(gs[1, 2])
    folds = walk_fwd_results["fold"]
    rmses = walk_fwd_results["rmse"]
    ax5.bar(folds, rmses, color=colors["test"], alpha=0.8, edgecolor="#555")
    ax5.axhline(rmses.mean(), color=colors["tuned"], lw=2, ls="--", label=f"Mean={rmses.mean():.5f}")
    ax5.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax5, " Walk-Forward RMSE per Fold")
    ax5.set_xlabel("Fold"); ax5.set_ylabel("RMSE")

    # ── 6. Test Set Predictions ──
    ax6 = fig.add_subplot(gs[2, :])
    tuned_pred_test = tuned_model.predict(X_test)
    ax6.plot(X_test.index, y_test.values,        color=colors["test"],  alpha=0.8, lw=1.2, label="Actual")
    ax6.plot(X_test.index, tuned_pred_test,       color=colors["tuned"], alpha=0.8, lw=1.2, label="Predicted (Tuned)")
    ax6.plot(X_test.index, baseline_model.predict(X_test), color=colors["base"], alpha=0.5, lw=0.8, ls="--", label="Predicted (Baseline)")
    ax6.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    ax6.axhline(0, color="#555", lw=0.5)
    style_ax(ax6, "Test Set: Actual vs Predicted Log Returns")
    ax6.set_ylabel("Log Return")

    # ── 7. Residuals ──
    ax7 = fig.add_subplot(gs[3, 0])
    residuals = y_test.values - tuned_pred_test
    ax7.hist(residuals, bins=40, color=colors["pred"], alpha=0.8, density=True)
    style_ax(ax7, "Residuals Distribution (Test)")
    ax7.set_xlabel("Residual")

    # ── 8. IC per walk-forward fold ──
    ax8 = fig.add_subplot(gs[3, 1])
    ics = walk_fwd_results["ic"]
    ax8.bar(folds, ics, color=colors["val"], alpha=0.8, edgecolor="#555")
    ax8.axhline(ics.mean(), color=colors["tuned"], lw=2, ls="--", label=f"Mean IC={ics.mean():.3f}")
    ax8.axhline(0, color="#555", lw=0.5)
    ax8.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax8, "Walk-Forward IC per Fold")
    ax8.set_xlabel("Fold"); ax8.set_ylabel("IC")

    # ── 9. Robustness: RMSE across seeds ──
    ax9 = fig.add_subplot(gs[3, 2])
    ax9.scatter(robustness_results["seed"], robustness_results["test_rmse"],
                color=colors["test"], s=60, zorder=3, label="Test RMSE")
    ax9.axhline(robustness_results["test_rmse"].mean(),
                color=colors["tuned"], lw=2, ls="--",
                label=f"Mean={robustness_results['test_rmse'].mean():.5f}")
    ax9.legend(fontsize=8, facecolor=ax_bg, edgecolor="#555", labelcolor="#e0e0e0")
    style_ax(ax9, "🔒 Robustness: RMSE across Seeds")
    ax9.set_xlabel("Random Seed"); ax9.set_ylabel("Test RMSE")

    # ── 10. Summary Table ──
    ax10 = fig.add_subplot(gs[4, :])
    ax10.set_facecolor(ax_bg)
    ax10.axis("off")

    base_tr  = compute_metrics(y_train, baseline_model.predict(X_train))
    base_vl  = compute_metrics(y_val,   baseline_model.predict(X_val))
    base_te  = compute_metrics(y_test,  baseline_model.predict(X_test))
    tuned_tr = compute_metrics(y_train, tuned_model.predict(X_train))
    tuned_vl = compute_metrics(y_val,   tuned_model.predict(X_val))
    tuned_te = compute_metrics(y_test,  tuned_model.predict(X_test))

    def pct_change(new, old):
        return f"({(new-old)/old*100:+.1f}%)"

    table_data = [
        ["Split", "Baseline RMSE", "Tuned RMSE", "Change",
         "Baseline IC", "Tuned IC", "Change"],
        ["Train",
         f"{base_tr['RMSE']:.6f}", f"{tuned_tr['RMSE']:.6f}",
         pct_change(tuned_tr['RMSE'], base_tr['RMSE']),
         f"{base_tr['IC']:.4f}",   f"{tuned_tr['IC']:.4f}",
         pct_change(tuned_tr['IC'], base_tr['IC'])],
        ["Val",
         f"{base_vl['RMSE']:.6f}", f"{tuned_vl['RMSE']:.6f}",
         pct_change(tuned_vl['RMSE'], base_vl['RMSE']),
         f"{base_vl['IC']:.4f}",   f"{tuned_vl['IC']:.4f}",
         pct_change(tuned_vl['IC'], base_vl['IC'])],
        ["Test",
         f"{base_te['RMSE']:.6f}", f"{tuned_te['RMSE']:.6f}",
         pct_change(tuned_te['RMSE'], base_te['RMSE']),
         f"{base_te['IC']:.4f}",   f"{tuned_te['IC']:.4f}",
         pct_change(tuned_te['IC'], base_te['IC'])],
    ]

    col_widths = [0.08, 0.14, 0.14, 0.10, 0.14, 0.14, 0.10]
    col_starts = [0.02]
    for w in col_widths[:-1]:
        col_starts.append(col_starts[-1] + w)

    header_color = "#2d3748"
    row_colors   = ["#1e2330", "#232840"]

    for r_idx, row in enumerate(table_data):
        for c_idx, (val, xs, w) in enumerate(zip(row, col_starts, col_widths)):
            bg = header_color if r_idx == 0 else row_colors[(r_idx-1) % 2]
            fc = ax10.transAxes
            rect = plt.Rectangle((xs, 0.75 - r_idx * 0.22), w - 0.005, 0.2,
                                   transform=fc, facecolor=bg, edgecolor="#333", lw=0.5)
            ax10.add_patch(rect)
            color = "#ffd700" if r_idx == 0 else (
                "#ff6b6b" if "+" in val and "Change" not in table_data[0][c_idx] else
                "#69db7c" if "-" in val and c_idx in [3, 6] else "#e0e0e0"
            )
            ax10.text(xs + w/2, 0.75 - r_idx * 0.22 + 0.1, val,
                      ha="center", va="center", transform=fc,
                      color=color, fontsize=8.5,
                      fontweight="bold" if r_idx == 0 else "normal")

    ax10.set_title(" Summary: Baseline vs Tuned Model", color="#e0e0e0",
                   fontsize=11, fontweight="bold", pad=10)

    # เปลี่ยนพาร์ทที่บันทึกรูปด้วย ให้สร้างแฟ้มเผื่อไว้
    # os.makedirs("/mnt/user-data/outputs", exist_ok=True)
    # plt.savefig("/mnt/user-data/outputs/catboost_gold_report.png",
    #             dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    # print(" Report saved to /mnt/user-data/outputs/catboost_gold_report.png")
    plt.close()


# ─────────────────────────────────────────────
# 11. EXPORT PREDICTIONS
# ─────────────────────────────────────────────
def export_predictions(model, X_test: pd.DataFrame, y_test: pd.Series, 
                       output_dir: str = "../../../data/processed/predictions") -> None:
    """
    ส่งออกผลลัพธ์การทำนายของ Test Set ไปยังไฟล์ CSV
    ประกอบด้วยคอลัมน์: Date, actual_return, pred_return
    """
    # 1. สร้างโฟลเดอร์ปลายทางถ้ายังไม่มี
    os.makedirs(output_dir, exist_ok=True)
    
    # 2. ทำนายผลและสร้าง DataFrame
    preds = model.predict(X_test)
    
    results_df = pd.DataFrame({
        "actual_return": y_test.values,
        "pred_return": preds
    }, index=X_test.index)
    
    # 3. จัดการคอลัมน์ Date (ดึงจาก Index)
    results_df = results_df.reset_index()
    results_df.rename(columns={"date": "Date"}, inplace=True)
    
    # 4. บันทึกเป็น CSV
    end_date_str = results_df["Date"].max().strftime("%Y%m%d")
    filename = f"catboost_predictions_{end_date_str}.csv"
    filepath = os.path.join(output_dir, filename)
    
    results_df.to_csv(filepath, index=False)
    print(f" Exported predictions successfully to: {filepath}")
    print(results_df.head())


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
def main():
    print("\n" + "═"*55)
    print("   CatBoost Gold Return Prediction Pipeline")
    print("═"*55)

    # 1. Data
    print("\n Generating synthetic gold data...")
    raw_df = generate_gold_data(n_days=2000)

    # 2. Feature Engineering
    print("  Engineering features (no leakage)...")
    df_features = engineer_features(raw_df)
    print(f"   → {len(df_features)} rows × {df_features.shape[1]} columns")

    # 3. Split
    X_train, y_train, X_val, y_val, X_test, y_test, feat_cols = \
        time_based_split(df_features)

    # 4. Baseline
    print(" Training baseline model...")
    baseline = train_baseline(X_train, y_train, X_val, y_val)

    # 5. Optuna Tuning
    print("\n Running Optuna hyperparameter search (50 trials)...")
    best_params, study = run_optuna_tuning(X_train, y_train, n_trials=50)

    # 6. Tuned Model
    print("\n Training tuned model...")
    tuned = train_tuned_model(best_params, X_train, y_train, X_val, y_val)

    print("\n Tuned Model Performance:")
    compute_metrics(y_train, tuned.predict(X_train), "Train")
    compute_metrics(y_val,   tuned.predict(X_val),   "Val  ")
    compute_metrics(y_test,  tuned.predict(X_test),  "Test ")

    # 7. Walk-Forward Validation
    print("\nRunning Walk-Forward Validation...")
    wf_results = walk_forward_validation(df_features, best_params)

    # 8. Robustness Check
    print(" Running Robustness Check...")
    rob_results = robustness_check(best_params, X_train, y_train,
                                    X_val, y_val, X_test, y_test)

    # 9. Report
    print("\n Generating full report...")
    create_full_report(baseline, tuned, study,
                       X_train, y_train, X_val, y_val, X_test, y_test,
                       wf_results, rob_results, feat_cols)

    # 10. Export Predictions
    print("\n Exporting Test Set Predictions...")
    # export_predictions(tuned, X_test, y_test)

    print("\n" + "═"*55)
    print("  PIPELINE COMPLETE")
    print("═"*55 + "\n")

    return tuned, best_params, wf_results, rob_results


if __name__ == "__main__":
    tuned_model, best_params, wf_results, rob_results = main()


═══════════════════════════════════════════════════════
   CatBoost Gold Return Prediction Pipeline
═══════════════════════════════════════════════════════

 Generating synthetic gold data...
  Engineering features (no leakage)...
   → 1747 rows × 61 columns

───────────────────────────────────────────────────────
  DATA SPLIT SUMMARY
───────────────────────────────────────────────────────
  Train: 2017-12-20 → 2022-08-25  (1222 rows)
  Val:   2022-08-26   → 2023-08-28    ( 262 rows)
  Test:  2023-08-29  → 2024-08-29   ( 263 rows)
  Features: 54
  Target std (train): 0.011884
───────────────────────────────────────────────────────

 Training baseline model...
 Baseline Model:
  [Train] RMSE=0.010440 | MAE=0.008320 | IC=0.7992 | HitRate=0.789
  [Val  ] RMSE=0.011912 | MAE=0.009452 | IC=0.1201 | HitRate=0.534

 Running Optuna hyperparameter search (50 trials)...

Best Params (RMSE=0.011867):
   bootstrap_type: Bayesian
   iterations: 806
   depth: 9
   learning_rate: 0.14912680568906123